# Citadel — Phase 2: Train the Oversight (on a frozen Commander)

Freeze the Phase-1 Commander. Train Oversight to produce structured critiques — picking the right decision, naming the weakness, and suggesting counter-proposals. Reward uses the Oversight-specific function in `reward.py` (correct_veto, critique_precision, counter_proposal_adopted_succeeded, lesson_utility).

In [ ]:
!pip install -q trl==0.11.4 transformers==4.45.2 peft==0.13.2 bitsandbytes==0.44.1 accelerate==1.0.1 matplotlib
!pip install -q openenv-core fastapi pydantic uvicorn openai

In [ ]:
import os, sys, json
from pathlib import Path
sys.path.insert(0, '/content/Citadel')  # adjust to your checkout path

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import GRPOConfig, GRPOTrainer
from datasets import Dataset

from oversight_env import OversightEnv
from models import OversightAction, IncidentAction, ACTION_NAMES, SYSTEM_NAMES
from inference import (
    OVERSIGHT_SYSTEM_PROMPT, format_oversight_observation, parse_oversight_response,
    COMMANDER_SYSTEM_PROMPT, format_commander_observation, parse_commander_response,
)

COMMANDER_CHECKPOINT = './checkpoints/commander/final'
OVERSIGHT_BASE = 'Qwen/Qwen2.5-3B-Instruct'

In [ ]:
# --- Load frozen Commander ---
cmd_tok = AutoTokenizer.from_pretrained(COMMANDER_CHECKPOINT)
cmd_model = AutoModelForCausalLM.from_pretrained(
    COMMANDER_CHECKPOINT, torch_dtype=torch.bfloat16, device_map='auto', trust_remote_code=True
)
cmd_model.eval()
for p in cmd_model.parameters():
    p.requires_grad = False

# --- Load trainable Oversight ---
ov_tok = AutoTokenizer.from_pretrained(OVERSIGHT_BASE)
if ov_tok.pad_token is None:
    ov_tok.pad_token = ov_tok.eos_token
ov_model = AutoModelForCausalLM.from_pretrained(
    OVERSIGHT_BASE, torch_dtype=torch.bfloat16, device_map='auto', trust_remote_code=True
)

In [ ]:
# --- Commander policy wrapper (uses frozen model) ---
def commander_policy(obs_dict, history):
    user_msg = format_commander_observation(obs_dict, step=0, history=history)
    prompt = cmd_tok.apply_chat_template([
        {'role': 'system', 'content': COMMANDER_SYSTEM_PROMPT},
        {'role': 'user', 'content': user_msg},
    ], tokenize=False, add_generation_prompt=True)
    inputs = cmd_tok(prompt, return_tensors='pt', truncation=True, max_length=1800).to(cmd_model.device)
    with torch.no_grad():
        out = cmd_model.generate(**inputs, max_new_tokens=300, temperature=0.4, do_sample=True, pad_token_id=cmd_tok.pad_token_id)
    text = cmd_tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return parse_commander_response(text)

In [ ]:
# --- Reward: run one OversightEnv.step and read oversight_reward from metadata ---
def oversight_reward_fn(prompts, completions, **kwargs):
    rewards = []
    for prompt, completion in zip(prompts, completions):
        # Parse the oversight action from the completion
        ov_action = parse_oversight_response(completion)

        # Rebuild the OversightEnv state from kwargs metadata
        meta = kwargs.get('env_state', [{}] * len(prompts))
        task_id = meta.get('task_id', 'easy_1') if isinstance(meta, dict) else 'easy_1'
        gen = meta.get('adversary_gen', 1) if isinstance(meta, dict) else 1
        env = OversightEnv(commander_policy=commander_policy)
        env.reset(task_id=task_id, adversary_gen=gen)
        next_obs = env.step(ov_action)
        rewards.append(float(next_obs.reward or 0.0))
    return rewards

In [ ]:
def build_oversight_prompts(n_per_condition=8):
    examples = []
    for task_id in ['easy_1', 'medium_1', 'hard_1']:
        for gen in [1, 2, 3]:
            for seed in range(n_per_condition):
                env = OversightEnv(commander_policy=commander_policy)
                oobs = env.reset(task_id=task_id, seed=seed, adversary_gen=gen)
                user_msg = format_oversight_observation(oobs.model_dump())
                prompt = ov_tok.apply_chat_template([
                    {'role': 'system', 'content': OVERSIGHT_SYSTEM_PROMPT},
                    {'role': 'user', 'content': user_msg},
                ], tokenize=False, add_generation_prompt=True)
                examples.append({'prompt': prompt, 'task_id': task_id, 'adversary_gen': gen, 'seed': seed})
    return Dataset.from_list(examples)

train_ds = build_oversight_prompts(n_per_condition=6)
print(f'Oversight training prompts: {len(train_ds)}')

In [ ]:
config = GRPOConfig(
    output_dir='./checkpoints/oversight',
    max_steps=200,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    num_generations=4,
    max_prompt_length=1800,
    max_completion_length=220,
    temperature=0.7,
    logging_steps=5,
    save_steps=50,
    bf16=True,
    report_to='none',
)

trainer = GRPOTrainer(
    model=ov_model,
    args=config,
    reward_funcs=[oversight_reward_fn],
    train_dataset=train_ds,
    processing_class=ov_tok,
)
trainer.train()
trainer.save_model('./checkpoints/oversight/final')

In [ ]:
# --- Reward curve ---
import matplotlib.pyplot as plt
log = trainer.state.log_history
rewards = [x.get('reward') for x in log if x.get('reward') is not None]
plt.figure(figsize=(10, 4))
plt.plot(rewards, color='C1')
plt.title('Oversight — Reward over GRPO steps')
plt.xlabel('logging step'); plt.ylabel('reward')
plt.savefig('./checkpoints/oversight/reward_curve.png', dpi=120, bbox_inches='tight')
plt.show()